In [1]:
# 2차 EDA를 위해 Netflix 콘텐츠 단위 데이터와 TMDB 메타데이터를 불러오기

import pandas as pd

In [3]:
# 2차 EDA에 사용할 Netflix 분석 데이터와 TMDB 메타데이터 불러오기

import pandas as pd

netflix_tv_df = pd.read_csv(
    "../DATA/PROCESSED/netflix_tv_analysis.csv"
)

tmdb_df = pd.read_csv(
    "../DATA/PROCESSED/tmdb_metadata.csv"
)

print("Netflix:", netflix_tv_df.shape)
print("TMDB:", tmdb_df.shape)

Netflix: (159, 13)
TMDB: (402, 15)


In [4]:
# Netflix TV 분석 데이터와 TMDB 메타데이터를 작품명 기준으로 병합

merged_df = netflix_tv_df.merge(
    tmdb_df,
    left_on="show_title",
    right_on="netflix_title",
    how="left"
)

print(merged_df.shape)

(159, 28)


In [5]:
# 병합 후 TMDB 정보가 빠진 작품 수를 확인

print("TMDB 미매칭:", merged_df["tmdb_id"].isna().sum())

TMDB 미매칭: 0


In [6]:
# 같은 show_title이 여러 시즌으로 중복되어 있는지 확인

print("show_title 중복:", merged_df["show_title"].duplicated().sum())

show_title 중복: 28


In [7]:
# 같은 show_title이 여러 행에 나타나는 중복 작품들을 확인

duplicate_titles = merged_df[
    merged_df["show_title"].duplicated(keep=False)
].sort_values("show_title")

print(
    duplicate_titles[
        [
            "content_key",
            "show_title",
            "category",
            "max_cumulative",
            "long_hit",
            "first_rank",
            "first_week_hours"
        ]
    ].to_string(index=False)
)

                content_key        show_title         category  max_cumulative  long_hit  first_rank  first_week_hours
        CoComelon: Season 1         CoComelon     TV (English)               1         0          10           9140000
        CoComelon: Season 3         CoComelon     TV (English)               5         1           9          11430000
        CoComelon: Season 4         CoComelon     TV (English)               5         1           8          15930000
        Cobra Kai: Season 1         Cobra Kai     TV (English)               2         0           8          13450000
        Cobra Kai: Season 2         Cobra Kai     TV (English)               1         0          10          19340000
        Control Z: Season 1         Control Z TV (Non-English)               1         0           7           8580000
        Control Z: Season 2         Control Z TV (Non-English)               3         0           2          25430000
            Elite: Season 1             Elite TV

한 행 = Netflix 시즌/콘텐츠

Netflix 성과
long_hit
first_rank
first_week_hours
initial_country_reach
max_cumulative

+

TMDB 작품 특성
genres
original_language
production_countries
number_of_seasons
number_of_episodes
vote_average
...

In [8]:
# Netflix 시즌 데이터 중 TMDB 메타데이터가 연결되지 않은 행이 있는지 확인

print("TMDB 미매칭:", merged_df["tmdb_id"].isna().sum())

TMDB 미매칭: 0


In [9]:
# TMDB 정보가 매칭되지 않은 Netflix 작품을 확인

merged_df.loc[
    merged_df["tmdb_id"].isna(),
    ["content_key", "show_title", "category"]
]

,content_key,show_title,category


In [10]:
# 같은 show_title이 여러 시즌으로 존재하는 작품 수를 확인

duplicate_show_count = (
    merged_df.loc[
        merged_df["show_title"].duplicated(keep=False),
        "show_title"
    ]
    .nunique()
)

print("여러 시즌이 있는 작품 수:", duplicate_show_count)

여러 시즌이 있는 작품 수: 17


In [11]:
# Netflix + TMDB 병합 데이터의 컬럼별 결측치 개수를 확인

merged_df.isna().sum()

content_key                0
show_title                 0
category                   0
first_week                 0
last_week                  0
first_cumulative           0
max_cumulative             0
right_censored             0
long_hit                   0
first_rank                 0
first_week_hours           0
log_first_week_hours       0
initial_country_reach      1
netflix_title              0
tmdb_id                    0
tmdb_type                  0
genres                     0
original_language          0
production_countries       6
release_date             158
first_air_date             1
runtime                  158
episode_run_time           1
number_of_seasons          1
number_of_episodes         1
vote_average               0
vote_count                 0
popularity                 0
dtype: int64

In [12]:
# 병합된 최종 데이터의 크기와 컬럼을 확인

print("데이터 크기:", merged_df.shape)
print("\n컬럼:")
print(merged_df.columns.tolist())

데이터 크기: (159, 28)

컬럼:
['content_key', 'show_title', 'category', 'first_week', 'last_week', 'first_cumulative', 'max_cumulative', 'right_censored', 'long_hit', 'first_rank', 'first_week_hours', 'log_first_week_hours', 'initial_country_reach', 'netflix_title', 'tmdb_id', 'tmdb_type', 'genres', 'original_language', 'production_countries', 'release_date', 'first_air_date', 'runtime', 'episode_run_time', 'number_of_seasons', 'number_of_episodes', 'vote_average', 'vote_count', 'popularity']


1. release_date 158, runtime 158은 정상

사용하는 Netflix 데이터 159개는 기본적으로 TV 콘텐츠이나, 앞에서 예외로 하나 존재.

Dave Chappelle: The Closer
Netflix → TV (English)
TMDB → movie

그래서 최종적으로는:

TV    158개
movie   1개

형태일 가능성이 높음.

따라서:

release_date 결측 158개
runtime 결측 158개

영화 1개에만 값이 있어서 생긴 정상적인 구조적 결측.

반대로:

first_air_date 결측 1
number_of_seasons 결측 1
number_of_episodes 결측 1

도 바로 그 movie 1개 때문일 가능성이 높다.

In [13]:
# 병합 데이터에서 TMDB 기준 movie와 tv 작품 수를 확인

merged_df["tmdb_type"].value_counts()

tmdb_type
tv       158
movie      1
Name: count, dtype: int64

In [14]:
# 초기 국가 도달 수가 누락된 Netflix 콘텐츠를 확인

merged_df.loc[
    merged_df["initial_country_reach"].isna(),
    ["content_key", "show_title", "category", "first_week"]
]

,content_key,show_title,category,first_week
97,Octonauts: Above & Beyond: Season 1,Octonauts: Above & Beyond,TV (English),2021-09-12


In [16]:
# 초기 국가 도달 수 누락 원인을 확인하기 위해 Netflix 국가별 원본 데이터를 불러오기

country_df = pd.read_csv(
    "../DATA/RAW/[엔터] Netflix Top 10 Weekly Dataset/all-weeks-countries.csv",
    encoding="latin1"
)

In [17]:
# 초기 국가 도달 수가 누락된 작품이 국가별 Netflix 데이터에 존재하는지 확인

octonauts_country = country_df[
    country_df["show_title"] == "Octonauts: Above & Beyond"
]

print(octonauts_country.shape)
print(octonauts_country.head(20).to_string(index=False))

(0, 8)
Empty DataFrame
Columns: [country_name, country_iso2, week, category, weekly_rank, show_title, season_title, cumulative_weeks_in_top_10]
Index: []


In [18]:
# 국가별 Netflix 데이터에 Octonauts 관련 작품이 다른 제목으로 저장되어 있는지 확인

octonauts_candidates = country_df[
    country_df["show_title"].str.contains(
        "Octonauts",
        case=False,
        na=False
    )
]

print(octonauts_candidates.shape)

print(
    octonauts_candidates[
        ["show_title", "season_title", "category"]
    ]
    .drop_duplicates()
    .to_string(index=False)
)

(0, 8)
Empty DataFrame
Columns: [show_title, season_title, category]
Index: []


Octonauts: Above & Beyond는 국가별 Netflix 데이터에 아예 존재하지 않아서 initial_country_reach를 계산할 수 없는 것.

그래서 이 값은:

0이 아님
→ 데이터 없음
→ NaN 유지

로 처리

In [19]:
# TMDB 제작국가 정보가 없는 작품 6개를 확인

missing_production_country = merged_df.loc[
    merged_df["production_countries"].isna(),
    ["show_title", "tmdb_id", "tmdb_type", "genres", "original_language"]
]

print(missing_production_country.to_string(index=False))

                                     show_title  tmdb_id tmdb_type                genres original_language
            Cocaine Cowboys: The Kings of Miami   129206        tv           Documentary                en
                                      Dive Club   131830        tv                 Drama                en
                                          Elves    70059        tv       Animation, Kids                en
                                  Gone for Good   128068        tv Crime, Drama, Mystery                fr
Monsters Inside: The 24 Faces of Billy Milligan   132105        tv    Documentary, Crime                en
          Top Secret UFO Projects: Declassified   129677        tv           Documentary                en


In [20]:
# TV 작품의 episode_run_time에서 빈 리스트("[]")로 저장된 작품 수를 확인

empty_runtime_count = (
    merged_df["episode_run_time"]
    .astype(str)
    .eq("[]")
    .sum()
)

print("episode_run_time 빈 리스트:", empty_runtime_count)

episode_run_time 빈 리스트: 77


### episode_run_time 데이터 품질 확인

TMDB의 TV 상세정보에서 `episode_run_time`이 빈 리스트(`[]`)로 제공되는 사례가 다수 확인되었다.

- 분석 대상 TV 콘텐츠: 158개
- 빈 리스트: 77개
- 약 49%의 러닝타임 정보가 부족함

따라서 `episode_run_time`은 핵심 EDA 변수에서 우선 제외하고,
필요 시 보조 분석 변수로만 활용한다.